<a href="https://colab.research.google.com/github/Andrey66Sudakov/web_scraping/blob/main/web_scraping_task_Sudakov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Установка библиотек
!pip install requests beautifulsoup4 pandas

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_quotes_by_tags(tags_list):
    """
    Функция парсит цитаты с сайта quotes.toscrape.com по списку тегов.
    """
    base_url = "https://quotes.toscrape.com"
    data_list = []
    seen_links = set() # Для удаления дубликатов

    for tag in tags_list:
        url = f"{base_url}/tag/{tag}/"
        try:
            response = requests.get(url)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Ошибка для тега '{tag}': {e}")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')
        quotes = soup.find_all('div', class_='quote')

        for quote in quotes:
            text_span = quote.find('span', class_='text')
            quote_text = text_span.get_text(strip=True) if text_span else ""

            author_small = quote.find('small', class_='author')
            author_name = author_small.get_text(strip=True) if author_small else ""

            link_a = quote.find('a', href=True)
            full_link = base_url + link_a['href'] if link_a else ""

            if full_link not in seen_links:
                seen_links.add(full_link)
                data_list.append({
                    'Автор (вместо даты)': author_name,
                    'Цитата (вместо заголовка)': quote_text,
                    'Ссылка на материал': full_link
                })

    df = pd.DataFrame(data_list)
    return df

In [10]:
# Задаем список "поисковых запросов" (тегов)
search_queries = ['love', 'inspirational', 'humor']

# Вызываем функцию
result_df = scrape_quotes_by_tags(search_queries)

# Выводим результат
if not result_df.empty:
    print(f"Найдено записей: {len(result_df)}")
    display(result_df.head(10))  # Показываем первые 10 строк
else:
    print("Данные не найдены или произошла ошибка.")

Найдено записей: 24


,Автор (вместо даты),Цитата (вместо заголовка),Ссылка на материал
0,André Gide,“It is better to be hated for what you are tha...,https://quotes.toscrape.com/author/Andre-Gide
1,Marilyn Monroe,“This life is what you make it. No matter what...,https://quotes.toscrape.com/author/Marilyn-Monroe
2,Bob Marley,"“You may not be her first, her last, or her on...",https://quotes.toscrape.com/author/Bob-Marley
3,Elie Wiesel,"“The opposite of love is not hate, it's indiff...",https://quotes.toscrape.com/author/Elie-Wiesel
4,Friedrich Nietzsche,"“It is not a lack of love, but a lack of frien...",https://quotes.toscrape.com/author/Friedrich-N...
5,Pablo Neruda,"“I love you without knowing how, or when, or f...",https://quotes.toscrape.com/author/Pablo-Neruda
6,James Baldwin,“Love does not begin and end the way we seem t...,https://quotes.toscrape.com/author/James-Baldwin
7,Jane Austen,“There is nothing I would not do for those who...,https://quotes.toscrape.com/author/Jane-Austen
8,Albert Einstein,“There are only two ways to live your life. On...,https://quotes.toscrape.com/author/Albert-Eins...
9,Thomas A. Edison,"“I have not failed. I've just found 10,000 way...",https://quotes.toscrape.com/author/Thomas-A-Ed...


На habr.com я бы искал заголовки статей в тегах (h2 class="post__title") или (a class="post__title_link"), а дату — в (span class="post__time")

Поскольку habr постоянно меняет верстку, я заменил дату на имя автора, заголовок на цитату (в учебных целях)

_______________________________________________________________________________

Дополнительное (не обязательное):
Вернулся к книгам.


In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_books_extended(pages_count=2):
    base_url = "https://books.toscrape.com"
    data_list = []
    seen_links = set()

    # Заголовки для обхода простой защиты
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

    for page in range(1, pages_count + 1):
        url = f"{base_url}/catalogue/page-{page}.html"

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
        except requests.RequestException:
            print(f"Страница {page} недоступна.")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        articles = soup.find_all('article', class_='product_pod')

        for article in articles:
            h3 = article.find('h3')
            link_tag = h3.find('a')
            book_title = link_tag['title']

            # Корректное формирование ссылки (убираем ../)
            relative_link = link_tag['href']
            clean_relative = relative_link.replace('../', '')
            book_link = f"{base_url}/{clean_relative}"

            if book_link in seen_links:
                continue
            seen_links.add(book_link)

            price_tag = article.find('p', class_='price_color')
            price_raw = price_tag.get_text(strip=True) if price_tag else ""
            # Убираем лишний символ Â, если он есть
            price = price_raw.replace('Â', '').strip()

            star_tag = article.find('p', class_='star-rating')
            rating_text = star_tag['class'][1] if star_tag else "Zero"
            rating_num = rating_map.get(rating_text, 0)

            # Загрузка описания с заголовками
            description = ""
            try:
                desc_resp = requests.get(book_link, headers=headers)
                desc_resp.raise_for_status()
                desc_soup = BeautifulSoup(desc_resp.text, 'html.parser')

                # Поиск описания в мета-теге (самый надежный способ на этом сайте)
                meta_desc = desc_soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    description = meta_desc['content']
                else:
                    # Альтернативный поиск в блоке product_description
                    product_desc = desc_soup.find('div', id='product_description')
                    if product_desc:
                        p_tag = product_desc.find_next_sibling('p')
                        description = p_tag.get_text(strip=True) if p_tag else ""

                time.sleep(0.5) # Пауза 0.5 сек между запросами описаний

            except Exception as e:
                description = "Ошибка загрузки"

            data_list.append({
                'Цена (вместо даты)': price,
                'Название (заголовок)': book_title,
                'Ссылка на материал': book_link,
                'Текст материала (Описание)': description[:150] + "..." if len(description) > 150 else description,
                'Рейтинг': rating_num
            })

        time.sleep(1) # Пауза между страницами

    return pd.DataFrame(data_list)

In [18]:
# Запускаем парсинг 2 страниц книг
print("Запуск дополнительной части (парсинг книг)...")
books_df = scrape_books_extended(pages_count=2)

if not books_df.empty:
    print(f"Найдено книг: {len(books_df)}")
    display(books_df.head())
else:
    print("Книги не найдены.")

Запуск дополнительной части (парсинг книг)...
Найдено книг: 40


,Цена (вместо даты),Название (заголовок),Ссылка на материал,Текст материала (Описание),Рейтинг
0,£51.77,A Light in the Attic,https://books.toscrape.com/a-light-in-the-atti...,Ошибка загрузки,3
1,£53.74,Tipping the Velvet,https://books.toscrape.com/tipping-the-velvet_...,Ошибка загрузки,1
2,£50.10,Soumission,https://books.toscrape.com/soumission_998/inde...,Ошибка загрузки,1
3,£47.82,Sharp Objects,https://books.toscrape.com/sharp-objects_997/i...,Ошибка загрузки,4
4,£54.23,Sapiens: A Brief History of Humankind,https://books.toscrape.com/sapiens-a-brief-his...,Ошибка загрузки,5
